# Train on the Figshare 141-point splits

Train one XASBlock per element from the Figshare train split. Early stopping selects each checkpoint using Figshare validation loss. The final table reports Figshare validation, Figshare test, and the original Materials Cloud test results. Each element gets separate 3×3 plots for Figshare and original test spectra; panels show spectra near the 10th through 90th percentile of per-spectrum test MSE.

The new split contains target spectra and IDs; this notebook computes the matching 64D features from its POSCAR files with the pinned local M3GNet model. Start Jupyter from this repository or `tutorial_omnixas`. Edit `DATA_ROOT` or `RUN_NAME` below if needed. The first run featurizes all new split structures and can take a while.


In [ ]:
from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pymatgen.core import Structure
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
if (cwd / 'tutorial_omnixas').is_dir():
    REPO_ROOT = cwd
elif cwd.name == 'tutorial_omnixas' and (cwd.parent / 'omnixas').is_dir():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError('Start Jupyter from the OmniXAS repository root or tutorial_omnixas directory.')
DATA_ROOT = REPO_ROOT.parent / 'OmniXAS_data'
NEW_ROOT = DATA_ROOT / 'figshare_141_splits'
OLD_ROOT = REPO_ROOT / 'tutorial_omnixas'
ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
SPLITS = ['train', 'val', 'test']
RUN_NAME = 'figshare_141_seed42'
OUTPUT_ROOT = REPO_ROOT / 'output' / 'training' / 'figshare_141_pipeline'
RUN_DIR = OUTPUT_ROOT / RUN_NAME
SEED = 42

required = [
    REPO_ROOT / 'models' / 'M3GNet-MP-2021.2.8-PES' / 'model.pt',
    REPO_ROOT / 'omnixas' / 'featurizer' / 'm3gnet_featurizer.py',
]
for root in (NEW_ROOT / 'material_id_and_site', NEW_ROOT / 'ml_data', OLD_ROOT / 'material_id_and_site', OLD_ROOT / 'ml_data'):
    required.append(root)
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Required input is missing: {path}')
if RUN_DIR.exists():
    raise FileExistsError(f'Run directory already exists; change RUN_NAME to preserve prior results: {RUN_DIR}')

def read_ids(path):
    rows = [line.strip() for line in path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
    if not rows:
        raise ValueError(f'Empty ID split: {path}')
    return rows

def read_matrix(path, cols, label):
    if not path.is_file():
        raise FileNotFoundError(f'Missing {label}: {path}')
    array = np.loadtxt(path, dtype=np.float32, ndmin=2)
    if array.shape[1] != cols or not np.isfinite(array).all():
        raise ValueError(f'{label} must be finite with {cols} columns; got {array.shape}: {path}')
    return array

# Validate all split files and exact ID/target/feature row alignment before loading M3GNet.
new_data, old_data = {}, {}
for element in ELEMENTS:
    for split in SPLITS:
        key = (element, split)
        new_ids = read_ids(NEW_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt')
        new_y = read_matrix(NEW_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt', 141, 'new targets')
        if len(new_ids) != len(new_y):
            raise ValueError(f'New ID/target row mismatch for {key}: {len(new_ids)} != {len(new_y)}')
        old_ids = read_ids(OLD_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt')
        old_y = read_matrix(OLD_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt', 141, 'old targets')
        if len(old_ids) != len(old_y):
            raise ValueError(f'Old ID/target row mismatch for {key}: {len(old_ids)}, {len(old_y)}')
        old_data[key] = {'ids': old_ids, 'y': old_y}
        new_data[key] = {'ids': new_ids, 'y': new_y}
        for value in new_ids:
            try:
                material_id, site = value.rsplit('_', 1)
                int(site)
            except ValueError as exc:
                raise ValueError(f'Invalid new split site ID {value!r} in {key}') from exc
            poscar = DATA_ROOT / 'figshare' / 'FEFF' / element / material_id / 'FEFF-XANES' / f'{int(site):03d}_{element}' / 'POSCAR'
            if not poscar.is_file():
                raise FileNotFoundError(f'No POSCAR for {value}: {poscar}')
    print(f'{element}: new train/val/test = {[len(new_data[(element, s)]["ids"]) for s in SPLITS]}, old = {[len(old_data[(element, s)]["ids"]) for s in SPLITS]}')

if RUN_DIR.exists():
    raise FileExistsError(f'Run directory was created while preparing inputs: {RUN_DIR}')


## Train the Figshare encoder and heads

This cell runs the same scratch-encoder, UniversalXAS, and tuned-head pipeline as the showcase notebook. It uses the Figshare structures and targets.


In [ ]:
import runpy

SCRIPT = REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py'
OUTPUT_ROOT = REPO_ROOT / 'output' / 'training' / 'figshare_141_pipeline'
RUN_NAME = 'figshare_141_seed42'
RUN_DIR = OUTPUT_ROOT / RUN_NAME

# Keep graph batches small enough for a 24 GB GPU. The encoder batch is 8 * 16 = 128 sites.
script_args = [
    str(SCRIPT),
    '--dataset-root', str(NEW_ROOT),
    '--raw-root', str(DATA_ROOT / 'figshare'),
    '--output-root', str(OUTPUT_ROOT),
    '--run-name', RUN_NAME,
    '--precision', 'bf16-mixed',
    '--encoder-rows-per-element', '16',
    '--encoder-eval-batch-size', '64',
    '--num-workers', '16',
    '--batch-size', '4096',
    '--overwrite',
]
previous_argv = sys.argv
sys.argv = script_args
try:
    runpy.run_path(str(SCRIPT), run_name='__main__')
finally:
    sys.argv = previous_argv


## Evaluate the trained encoder and heads

The previous cell trains the scratch encoder, UniversalXAS head, and tuned heads. This cell evaluates both head types on Figshare and evaluates the tuned heads on the original test split.


In [ ]:
from torch.utils.data import DataLoader
from omnixas.data.feff_graph import CollateGraphs, FEFFDataset, FEFF_TASKS
from omnixas.model.m3gnet_xas import M3GNetXAS, XASSpectralHead

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for element in ELEMENTS:
    task = f'{element}_FEFF'
    for split in SPLITS:
        data = new_data[(element, split)]
        data['X'] = np.atleast_2d(np.loadtxt(RUN_DIR / 'features' / f'{task}_{split}_X.txt', dtype=np.float32))
        if data['X'].shape != (len(data['ids']), 64):
            raise ValueError(f'Invalid trained Figshare features for {task} {split}: {data["X"].shape}')

encoder = M3GNetXAS()
checkpoint = torch.load(RUN_DIR / 'best_encoder.ckpt', map_location='cpu', weights_only=False)
state = checkpoint['state_dict']
encoder.load_state_dict({k.removeprefix('model.'): v for k, v in state.items() if k.startswith('model.')}, strict=True)
encoder.eval().to(device)

old_raw = DATA_ROOT / 'materialscloud_omnixas_raw' / 'extracted'
old_features = {}
collate = CollateGraphs(encoder.encoder)
for element in ELEMENTS:
    task = f'{element}_FEFF'
    for split in SPLITS:
        dataset = FEFFDataset(REPO_ROOT, old_raw, [task], split, OLD_ROOT)
        loader = DataLoader(dataset, batch_size=512, shuffle=False, collate_fn=collate)
        rows = []
        with torch.inference_mode():
            for batch in loader:
                rows.append(encoder.encode(batch['graph'].to(device), batch['line_graph'].to(device), batch['site'].to(device), scaled=True).cpu().numpy())
        X = np.concatenate(rows, axis=0)
        old_data[(element, split)]['X'] = X
        old_features[(element, split)] = X

def load_head(path):
    head = XASSpectralHead().to(device)
    head.load_state_dict(torch.load(path, map_location='cpu', weights_only=False)['state_dict'], strict=True)
    return head.eval()

heads = {
    'UniversalXAS': load_head(RUN_DIR / 'heads' / 'universalXAS' / 'best.pt'),
    'Tuned-UniversalXAS': {element: load_head(RUN_DIR / 'heads' / 'tunedUniversalXAS' / f'{element}_FEFF' / 'best.pt') for element in ELEMENTS},
}

def predict(head, X):
    output = []
    with torch.inference_mode():
        for start in range(0, len(X), 4096):
            output.append(head(torch.as_tensor(X[start:start + 4096], dtype=torch.float32, device=device)).cpu().numpy())
    return np.concatenate(output, axis=0)

def score(element, variant, split_name, y_true, y_pred, train_mean):
    errors = np.mean((y_true - y_pred) ** 2, axis=1)
    baseline = np.mean((y_true - train_mean) ** 2, axis=1)
    return {
        'element': element, 'variant': variant, 'evaluation_split': split_name, 'n_spectra': len(y_true),
        'aggregate_mse': float(np.mean((y_true - y_pred) ** 2)), 'median_spectrum_mse': float(np.median(errors)),
        'baseline_median_mse': float(np.median(baseline)), 'eta': float(np.median(baseline) / max(np.median(errors), 1e-12)),
    }, errors

records, predictions = [], {}
for element in ELEMENTS:
    train = new_data[(element, 'train')]
    train_mean = train['y'].mean(axis=0, keepdims=True)
    for variant, head_source in heads.items():
        head = head_source if variant == 'UniversalXAS' else head_source[element]
        for split_name, data in [('new_val', new_data[(element, 'val')]), ('new_test', new_data[(element, 'test')]), ('old_test', old_data[(element, 'test')])]:
            pred = predict(head, data['X'])
            row, errors = score(element, variant, split_name, data['y'], pred, train_mean)
            records.append(row)
            if variant == 'Tuned-UniversalXAS':
                predictions[(element, split_name)] = {'y': data['y'], 'pred': pred, 'errors': errors}

results = pd.DataFrame(records).sort_values(['element', 'variant', 'evaluation_split'])
results.to_csv(RUN_DIR / 'evaluation_with_old_test.csv', index=False)
display(results.round({'aggregate_mse': 6, 'median_spectrum_mse': 6, 'baseline_median_mse': 6, 'eta': 3}))

## Test spectrum percentiles

Each panel shows the target and tuned-head prediction for the spectrum nearest that error percentile, with spectra ranked by per-spectrum MSE. Separate figures keep the new and original test sets easy to compare.


In [ ]:
percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90]
energy = np.arange(141) * 0.25
for element in ELEMENTS:
    for split_name in ('new_test', 'old_test'):
        item = predictions[(element, split_name)]
        order = np.argsort(item['errors'])
        fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True, sharey=True)
        for ax, percentile in zip(axes.flat, percentiles):
            idx = order[int(round((percentile / 100) * (len(order) - 1)))]
            ax.plot(energy, item['y'][idx], label='target', linewidth=1.5)
            ax.plot(energy, item['pred'][idx], label='prediction', linewidth=1.2)
            ax.set_title(f'{percentile}th percentile (MSE={item["errors"][idx]:.3g})')
            ax.grid(alpha=0.2)
        fig.suptitle(f'{element} — {split_name.replace("_", " ")}')
        fig.supxlabel('Energy above grid start (eV)')
        fig.supylabel('Intensity')
        axes.flat[0].legend(frameon=False)
        fig.tight_layout()
        fig.savefig(RUN_DIR / f'{element}_{split_name}_percentiles.png', dpi=140, bbox_inches='tight')
        plt.show()
        plt.close(fig)
print(f'Results and plots saved in {RUN_DIR}')